# MOD-12930 addendum — balanced branches

The main matrix never balances the two subqueries: text is either near-zero (selective)
or dominant (broad). This addendum **calibrates text selectivity until the SEARCH mirror's
p50 matches the VSIM mirror's** (±20%, geometric bisection on the target match count),
then measures the same four contenders in that regime. fields=none, depth K/W=1000/2000,
sizes 100K/500K (at 10K the text branch can't reach vector-branch latency even matching
the whole corpus).

**What this regime uniquely shows:**
- **Parallelism gain** `t(w0)/t(w6)` — with equal branches, truly concurrent depletion
  approaches 2×; anything well below that bounds what workers actually buy.
- **ε with no dominant branch** — merger + YIELD overhead, not masked by branch skew.

In [1]:
import json
import pandas as pd
import bench_lib as B
import balanced_lib as BAL

titles, texts, emb, corpus_max = B.load_data()
results, gates, meta = BAL.run_balanced(titles, texts, emb, corpus_max)

with open('results_balanced.json', 'w') as f:
    json.dump(dict(meta=meta, results=results, gates=gates), f, indent=2, default=str)
print('saved results_balanced.json')


===== balanced, dataset size 100000 =====


loaded+indexed 100000 docs in 52.4s (hash_indexing_failures=0)


n=100000: vsim mirror p50 = 9.07ms — calibrating text to match


  target≈4,472 matches -> search p50 1.48ms


  target≈21,147 matches -> search p50 6.68ms


...

n=500000 w=0 vsim_branch    qps=   111.5 p50=9.09ms p99=11.58ms


n=500000 w=6 hybrid_linear  qps=    55.7 p50=17.23ms p99=28.62ms


n=500000 w=6 hybrid_rrf     qps=    57.9 p50=16.36ms p99=27.74ms


n=500000 w=6 search_branch  qps=   121.7 p50=6.63ms p99=19.11ms


n=500000 w=6 vsim_branch    qps=   115.1 p50=8.85ms p99=10.74ms
saved results_balanced.json


## Gates & calibration

In [2]:
print(json.dumps(meta['calibration'], indent=1))
pd.DataFrame(gates)

[
 {
  "size": 100000,
  "vsim_p50_ms": 9.07,
  "search_p50_ms": 9.14,
  "target": 37869,
  "matches_mean": 43318.625,
  "matches_median": 43592.0
 },
 {
  "size": 500000,
  "vsim_p50_ms": 9.13,
  "search_p50_ms": 7.15,
  "target": 51043,
  "matches_mean": 57518.40625,
  "matches_median": 56985.0
 }
]


,size,matches_mean,gate_linear,gate_rrf
0,100000,43318.62500,16/16,16/16
1,500000,57518.40625,16/16,16/16


## Results — p50 (ms) and parallelism gain

`gain = p50(workers=0) / p50(workers=6)` per contender. For the hybrids, the ceiling with perfectly overlapping equal branches is ≈2× (minus merger/YIELD, which don't parallelize); the mirrors are single queries, so their gain should be ≈1.

In [3]:
df = pd.DataFrame(results)
p = df.pivot_table(index=['size', 'contender'], columns='workers', values='p50_ms', sort=False)
p.columns = [f'w{c}_p50_ms' for c in p.columns]
p['gain_w0/w6'] = (p['w0_p50_ms'] / p['w6_p50_ms']).round(2)
order = ['hybrid_linear', 'hybrid_rrf', 'search_branch', 'vsim_branch']
p = p.reindex([(s, c) for s in sorted(df['size'].unique()) for c in order])
p.round(2)

w0_p50_ms  w6_p50_ms  gain_w0/w6
size   contender                                      
100000 hybrid_linear      25.63      15.98        1.60
       hybrid_rrf         25.64      17.30        1.48
       search_branch      10.33      10.01        1.03
       vsim_branch         8.38       8.66        0.97
500000 hybrid_linear      26.11      17.23        1.52
       hybrid_rrf         26.69      16.36        1.63
       search_branch       8.09       6.63        1.22
       vsim_branch         9.09       8.85        1.03

### ε in the balanced regime

In [4]:
m = df.pivot_table(index=['size', 'workers'], columns='contender', values='mean_ms', sort=False)
eps = m['hybrid_linear'] - m[['search_branch', 'vsim_branch']].max(axis=1)
pd.DataFrame({'hybrid_mean_ms': m['hybrid_linear'].round(2),
              'slowest_branch_ms': m[['search_branch', 'vsim_branch']].max(axis=1).round(2),
              'sum_branches_ms': m[['search_branch', 'vsim_branch']].sum(axis=1).round(2),
              'eps_vs_max_ms': eps.round(2)})

hybrid_mean_ms  slowest_branch_ms  sum_branches_ms  \
size   workers                                                       
100000 0                 25.75              10.84            19.21   
       6                 16.35              10.36            19.01   
500000 0                 27.07               9.15            18.12   
       6                 17.95               8.69            16.91   

                eps_vs_max_ms  
size   workers                 
100000 0                14.91  
       6                 5.99  
500000 0                17.92  
       6                 9.26